## Transform and Join: Weitere Transformationen und Join

Dieses Notebook tansformiert EMDAT und Sea-Level weiter und führt den Join der Datensätze durch. Das Ergebnis ist `flood_linked`, eine analysefertige Tabelle unter anderem mit Meeresspiegel-Werten pro Flood-Ereignis.

In [30]:
import logging

import numpy as np
import pandas as pd

from myproj.pipeline import run_import, run_transform, run_cleaning

pd.set_option("display.max_columns", 60)

SEA_VALUE = "MSL_filtered_GIA_corrected_adjusted"
SEA_TREND = "trend_MSL_filtered_GIA_corrected_adjusted"
MED_ISO = [
    "ALB", "DZA", "BIH", "HRV", "CYP", "EGY", "FRA", "GRC", "ISR", "ITA",
    "LBN", "LBY", "MLT", "MNE", "MAR", "PSE", "SVN", "ESP", "SYR", "TUN", "TUR",
]

logger = logging.getLogger("myproj.transform")

emdat_raw, sea_level_raw = run_import()
emdat, sea_level = run_transform(emdat_raw, sea_level_raw)
emdat, sea_level = run_cleaning(emdat, sea_level)

print(f"EMDAT nach Pipeline: {emdat.shape[0]:,} Zeilen, {emdat.shape[1]} Spalten")
print(f"Sea-Level nach Pipeline: {sea_level.shape[0]:,} Zeilen, {sea_level.shape[1]} Spalten")

2026-04-29 16:49:57 | myproj.pipeline      | INFO     | run_import | start
2026-04-29 16:49:57 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-04-29 16:50:01 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-04-29 16:50:01 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-04-29 16:50:01 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-04-29 16:50:01 | myproj.pipeline      | INFO     | run_import | done
2026-04-29 16:50:01 | myproj.pipeline      | INFO     | run_transform | start
2026-04-29 16:50:01 | myproj.transform     | INFO     | select_columns | cols: 47 → 18
2026-04-29 16:50:01 | myproj.transform     | INFO     | filter_before_sea_level_start | rows: 20657 → 16769 | removed: 3888
202

EMDAT nach Pipeline: 16,769 Zeilen, 21 Spalten
Sea-Level nach Pipeline: 9,405 Zeilen, 3 Spalten


## Weitere Transformationen

### 2.1 Disaster-Type-Filter → nur Floods

**Problem:** EMDAT enthält alle Katastrophentypen. Für das Projektziel (Einfluss des Meeresspiegels auf Überschwemmungen) sind nur Flood-Ereignisse relevant.

**Begründung:** Sea-Level-Werte sollen gezielt Flood-Ereignissen zugeordnet werden. Andere Katastrophentypen würden die Analyse verfälschen.

**Implementierung:** Textfilter (case-insensitive) auf `Disaster Type` ODER `Disaster Subtype`, um auch Einträge zu erfassen, bei denen nur der Subtyp "Flood" enthält.

In [31]:
emdat["Disaster Type"].value_counts()

Disaster Type
Flood                               4240
Storm                               2770
Road                                2288
Water                               1167
Epidemic                             939
Earthquake                           693
Extreme temperature                  573
Mass movement (wet)                  489
Explosion (Industrial)               476
Air                                  460
Fire (Miscellaneous)                 449
Drought                              435
Wildfire                             333
Rail                                 266
Miscellaneous accident (General)     203
Collapse (Miscellaneous)             172
Explosion (Miscellaneous)            170
Collapse (Industrial)                153
Volcanic activity                    134
Fire (Industrial)                    116
Industrial accident (General)        103
Gas leak                              36
Infestation                           29
Chemical spill                        24
Po

In [32]:
def filter_to_floods(df: pd.DataFrame) -> pd.DataFrame:
    mask = (
        df["Disaster Type"].str.contains("flood", case=False, na=False)
        | df["Disaster Subtype"].str.contains("flood", case=False, na=False)
    )
    result = df[mask].copy()
    logger.info(
        "filter_to_floods | rows: %d → %d | removed: %d",
        len(df), len(result), len(df) - len(result),
    )
    return result


emdat_flood = filter_to_floods(emdat)
print(f"Flood-Ereignisse: {len(emdat_flood):,}")

2026-04-29 16:50:01 | myproj.transform     | INFO     | filter_to_floods | rows: 16769 → 4245 | removed: 12524


Flood-Ereignisse: 4,245


### 2.2 Geographischer Filter → Mittelmeerraum

**Problem:** EMDAT ist global. Der Sea-Level-Datensatz misst den Meeresspiegel im Mittelmeerraum. Länder ausserhalb des Mittelmeerraums haben keinen Bezug zu diesem Meeresspiegel, ihre Einbeziehung würde die Analyse verfälschen.

**Begründung:** Nur Ereignisse in Mittelmeerländern können sinnvoll mit dem Sea-Level-Datensatz verknüpft werden.

**Implementierung:** Filter auf vordefinierte ISO-Ländercodes des Mittelmeerraums (`MED_ISO`).

In [33]:
print(f"Länder vor Filter: {emdat_flood['ISO'].nunique()}")
print(emdat_flood["ISO"].value_counts().head(10))

Länder vor Filter: 180
ISO
CHN    227
IND    204
IDN    203
USA    119
BRA    116
PHL    111
AFG     91
PAK     90
VNM     87
COL     78
Name: count, dtype: int64


In [34]:
def filter_to_mediterranean(df: pd.DataFrame, iso_codes: list[str]) -> pd.DataFrame:
    result = df[df["ISO"].isin(iso_codes)].copy()
    logger.info(
        "filter_to_mediterranean | rows: %d → %d | removed: %d",
        len(df), len(result), len(df) - len(result),
    )
    return result


emdat_flood = filter_to_mediterranean(emdat_flood, MED_ISO)
print(f"Mittelmeer-Flood-Ereignisse: {len(emdat_flood):,}")

2026-04-29 16:50:01 | myproj.transform     | INFO     | filter_to_mediterranean | rows: 4245 → 293 | removed: 3952


Mittelmeer-Flood-Ereignisse: 293


### 2.3 Datum konstruieren

**Problem:** EMDAT speichert das Datum als separate Float-Spalten (`Start Year`, `Start Month`, `Start Day`). Für den Sea-Level-Join werden datetime-Objekte benötigt.

**Begründung:** Fehlende Monate werden mit 1 (→ Januar), fehlende Tage mit 1 aufgefüllt. Diese Approximation ist unvermeidlich und wird durch `start_date_quality` transparent dokumentiert. Beim Interpretieren von `sea_level_at_start` muss berücksichtigt werden, dass Ereignisse mit `month_and_day_imputed` ein ungenaueres Datum haben.

**Implementierung:** Hilfsfunktionen `make_event_date` und `date_quality`, Hauptfunktion `add_event_dates` mit Logging.

In [35]:
emdat_flood[["Start Year", "Start Month", "Start Day"]].dtypes

Start Year       int64
Start Month    float64
Start Day      float64
dtype: object

In [36]:
def make_event_date(df: pd.DataFrame, prefix: str) -> pd.Series:
    return pd.to_datetime(
        pd.DataFrame({
            "year":  pd.to_numeric(df[f"{prefix} Year"],  errors="coerce"),
            "month": pd.to_numeric(df[f"{prefix} Month"], errors="coerce").fillna(1),
            "day":   pd.to_numeric(df[f"{prefix} Day"],   errors="coerce").fillna(1),
        }),
        errors="coerce",
    )


def date_quality(df: pd.DataFrame, prefix: str) -> pd.Series:
    month_missing = df[f"{prefix} Month"].isna()
    day_missing   = df[f"{prefix} Day"].isna()
    return np.select(
        [month_missing & day_missing, month_missing, day_missing],
        ["month_and_day_imputed", "month_imputed", "day_imputed"],
        default="complete",
    )


def add_event_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["start_date"]         = make_event_date(df, "Start")
    df["end_date"]           = make_event_date(df, "End")
    df["start_date_quality"] = date_quality(df, "Start")
    df["end_date_quality"]   = date_quality(df, "End")
    n_complete = (df["start_date_quality"] == "complete").sum()
    n_day      = (df["start_date_quality"] == "day_imputed").sum()
    n_both     = (df["start_date_quality"] == "month_and_day_imputed").sum()
    logger.info(
        "add_event_dates | rows: %d | quality: complete=%d, day_imputed=%d, month_and_day_imputed=%d",
        len(df), n_complete, n_day, n_both,
    )
    return df


emdat_flood = add_event_dates(emdat_flood)
print(emdat_flood[["start_date", "start_date_quality", "end_date", "end_date_quality"]].head())

2026-04-29 16:50:01 | myproj.transform     | INFO     | add_event_dates | rows: 293 | quality: complete=291, day_imputed=2, month_and_day_imputed=0


     start_date start_date_quality   end_date end_date_quality
4198 1999-11-12           complete 1999-11-15         complete
5036 2000-10-31           complete 2000-10-31         complete
5099 2023-12-30           complete 2024-01-03         complete
5191 2022-12-11           complete 2022-12-11         complete
5192 2022-12-11           complete 2022-12-22         complete


### 2.4 Zeitlicher Cutoff (obere Grenze)

**Problem:** `filter_before_sea_level_start` in der Pipeline entfernt nur Ereignisse **vor** 1999-02-20. Ereignisse nach dem letzten Sea-Level-Messwert würden nach dem Join NaN in allen Sea-Level-Spalten erhalten.

**Begründung:** Der Join soll vollständig sein. Nur Ereignisse innerhalb der Sea-Level-Datensatzabdeckung werden behalten.

**Implementierung:** Beidseitiger Filter auf `start_date`.

In [37]:
def filter_to_sea_level_coverage(
    df: pd.DataFrame,
    coverage_start: pd.Timestamp,
    coverage_end: pd.Timestamp,
) -> pd.DataFrame:
    mask = df["start_date"].between(coverage_start, coverage_end)
    result = df[mask].copy()
    logger.info(
        "filter_to_sea_level_coverage | rows: %d → %d | removed: %d | coverage: %s – %s",
        len(df), len(result), len(df) - len(result),
        coverage_start.date(), coverage_end.date(),
    )
    return result


sea_coverage_start = pd.to_datetime(sea_level["time"]).min()
sea_coverage_end   = pd.to_datetime(sea_level["time"]).max()
emdat_flood = filter_to_sea_level_coverage(emdat_flood, sea_coverage_start, sea_coverage_end)
print(f"Ereignisse nach Cutoff: {len(emdat_flood):,}")

2026-04-29 16:50:01 | myproj.transform     | INFO     | filter_to_sea_level_coverage | rows: 293 → 292 | removed: 1 | coverage: 1999-02-20 – 2024-11-19


Ereignisse nach Cutoff: 292


### 2.5 Abgeleitete Spalten

Ergänzung um analyserelevante Spalten, keine Bereinigung, sondern Feature-Erweiterung.

- **`season`**: Saisonzuordnung aus dem Startmonat. Saison ist ein möglicher Confounder, da sie sowohl Niederschlagsintensität als auch Sea-Level-Anomalien beeinflusst.
- **`event_duration_days`**: Dauer in Tagen aus `end_date − start_date + 1`. Beeinflusst, wie aussagekräftig `mean_sea_level_while_disaster` ist.
- **`has_coordinates`**: Bool-Flag aus Latitude/Longitude. Für spätere geospatiale Auswertungen.

In [38]:
def season_from_month(month: float) -> str:
    if pd.isna(month):
        return "unknown"
    m = int(month)
    if m in [12, 1, 2]:
        return "winter"
    if m in [3, 4, 5]:
        return "spring"
    if m in [6, 7, 8]:
        return "summer"
    return "autumn"


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["season"]              = df["Start Month"].apply(season_from_month)
    df["event_duration_days"] = ((df["end_date"] - df["start_date"]).dt.days + 1).clip(lower=1)
    df["has_coordinates"]     = df[["Latitude", "Longitude"]].notna().all(axis=1)
    logger.info(
        "add_derived_columns | added: season, event_duration_days, has_coordinates | rows: %d",
        len(df),
    )
    return df


emdat_flood = add_derived_columns(emdat_flood)
print(emdat_flood["season"].value_counts())

2026-04-29 16:50:01 | myproj.transform     | INFO     | add_derived_columns | added: season, event_duration_days, has_coordinates | rows: 292


season
autumn    117
winter     67
summer     56
spring     52
Name: count, dtype: int64


### 2.6 Sea Level vorbereiten

**Problem:** `sea_level["time"]` muss als sortierter DatetimeIndex vorliegen, damit `.map()` und DatetimeIndex-Slicing im Join korrekt funktionieren.

**Implementierung:**

In [39]:
sea_level_ts = sea_level.copy()
sea_level_ts["time"] = pd.to_datetime(sea_level_ts["time"])
sea_level_ts = sea_level_ts.sort_values("time").reset_index(drop=True)
print(
    f"Sea-Level: {len(sea_level_ts):,} Tageswerte | "
    f"{sea_level_ts['time'].min().date()} bis {sea_level_ts['time'].max().date()}"
)

Sea-Level: 9,405 Tageswerte | 1999-02-20 bis 2024-11-19


## Format-Entscheid und Join

### Format-Entscheid

| Tabelle | Aktuelles Format | Transformation nötig? | Grund |
|---|---|---|---|
| `emdat_flood` | **Wide**, eine Zeile pro Flood-Ereignis | Nein | Spalten werden nur ergänzt |
| `sea_level_ts` | **Long / Zeitreihe**, eine Zeile pro Tag | Nein | Dient als Lookup-Tabelle |
| `flood_linked` (Output) | **Wide** | Nein | Ereignis und Sea-Level-Spalten, direkt analysierbar |

Beide Tabellen sind bereits im richtigen Format. Der Join hängt Sea-Level-Spalten an `emdat_flood` an. Keine Format-Transformation nötig.

### Join-Strategie

**Join-Typ: Left Join.** `emdat_flood` ist die führende Tabelle. Alle Flood-Ereignisse werden behalten. Sea-Level-Werte werden über `.map()` auf einen DatetimeIndex angehängt. Fehlt der genaue Tag im Sea-Level-Datensatz, entsteht NaN, erklärt durch `start_date_quality`.

| Neue Spalte | Beschreibung |
|---|---|
| `sea_level_at_start` | Meeresspiegel am `start_date`, primäre Analysevariable |
| `sea_trend_at_start` | Langfristiger Trend am `start_date` (langsam steigender Meeresspiegel) |
| `sea_level_at_end` | Meeresspiegel am `end_date`, NaN wenn `end_date` fehlt |
| `mean_sea_level_while_disaster` | Mittlerer Meeresspiegel zwischen `start_date` und `end_date`. Sinnvoll bei langen Riverine-Floods. Bei kurzen Flash-Floods (~1 bis 3 Tage) ähnlich wie `sea_level_at_start` |
| `sea_level_lag_1d` bis `sea_level_lag_5d` | Sea-Level der 1 bis 5 Tage **vor** `start_date`. Erhöhter Meeresspiegel in den Vortagen kann Vorbedingungen wie Grundwasserspiegel oder Bodenfeuchte beeinflusst haben, analysierbar für zeitverzögerte Korrelation |

In [40]:
def _mean_sea_level(start: pd.Timestamp, end: pd.Timestamp, sea_msl_idx: pd.Series) -> float:
    if pd.isna(start) or pd.isna(end) or end < start:
        return float("nan")
    return sea_msl_idx.loc[start:end].mean()


def join_sea_level(emdat_flood: pd.DataFrame, sea_level_ts: pd.DataFrame) -> pd.DataFrame:
    df = emdat_flood.copy()
    sea_msl_idx   = sea_level_ts.set_index("time")[SEA_VALUE]
    sea_trend_idx = sea_level_ts.set_index("time")[SEA_TREND]

    df["sea_level_at_start"] = df["start_date"].map(sea_msl_idx)
    df["sea_trend_at_start"] = df["start_date"].map(sea_trend_idx)
    df["sea_level_at_end"]   = df["end_date"].map(sea_msl_idx)
    df["mean_sea_level_while_disaster"] = df.apply(
        lambda row: _mean_sea_level(row["start_date"], row["end_date"], sea_msl_idx),
        axis=1,
    )

    for n in range(1, 6):
        df[f"sea_level_lag_{n}d"] = (df["start_date"] - pd.Timedelta(days=n)).map(sea_msl_idx)

    n_at_start = int(df["sea_level_at_start"].notna().sum())
    n_mean     = int(df["mean_sea_level_while_disaster"].notna().sum())
    logger.info(
        "join_sea_level | events: %d | sea_level_at_start_not_null: %d | mean_sea_level_not_null: %d",
        len(df), n_at_start, n_mean,
    )
    return df


emdat_flood = join_sea_level(emdat_flood, sea_level_ts)


2026-04-29 16:50:01 | myproj.transform     | INFO     | join_sea_level | events: 292 | sea_level_at_start_not_null: 292 | mean_sea_level_not_null: 292


### Redundante Spalten entfernen

Nach dem Join sind einige Spalten konstant oder durch andere Spalten vollständig abgedeckt:

| Spalte | Grund |
|---|---|
| `Disaster Type` | Konstant "Flood" nach Filter, kein Informationsgehalt mehr |
| `Disaster Subgroup` | Konstant "Hydrological" nach Filter |
| `Start Year`, `Start Month`, `Start Day` | Durch `start_date` und `start_date_quality` ersetzt |
| `End Year`, `End Month`, `End Day` | Durch `end_date` und `end_date_quality` ersetzt |

`is_outlier`, `is_outlier_Total Affected` und `is_outlier_Total Deaths` (aus `flag_outliers` in der Pipeline) bleiben vollständig erhalten.

In [41]:
DROP_COLS = [
    "Disaster Type",
    "Disaster Subgroup",
    "Start Year", "Start Month", "Start Day",
    "End Year",   "End Month",   "End Day",
]
flood_linked = emdat_flood.drop(columns=DROP_COLS).reset_index(drop=True)
print(f"Spalten nach Bereinigung: {flood_linked.shape[1]}")
print(flood_linked.columns.tolist())

Spalten nach Bereinigung: 29
['DisNo.', 'ISO', 'Country', 'Region', 'Disaster Subtype', 'Origin', 'Latitude', 'Longitude', 'Total Affected', 'Total Deaths', 'is_outlier_Total Affected', 'is_outlier_Total Deaths', 'is_outlier', 'start_date', 'end_date', 'start_date_quality', 'end_date_quality', 'season', 'event_duration_days', 'has_coordinates', 'sea_level_at_start', 'sea_trend_at_start', 'sea_level_at_end', 'mean_sea_level_while_disaster', 'sea_level_lag_1d', 'sea_level_lag_2d', 'sea_level_lag_3d', 'sea_level_lag_4d', 'sea_level_lag_5d']


### Skalierung

**Problem:** `sea_level_at_start` liegt typischerweise zwischen −3 und +8 cm. Für die Analyse ist nicht der absolute Wert entscheidend, sondern: Wie anomal war der Meeresspiegel zum Zeitpunkt dieses Flood-Ereignisses relativ zur gesamten Messperiode?

**Begründung:** Der Z-Score (z = (x − μ) / σ) ist eine **lineare Transformation**. Relative Unterschiede und Reihenfolge bleiben identisch, die Daten werden nicht verzerrt. Als Referenzverteilung dient die gesamte Sea-Level-Zeitreihe (nicht nur die Flood-Events), weil wir die Anomalie relativ zur historischen Norm einordnen wollen.

Die Rohwerte (`sea_level_at_start` und `mean_sea_level_while_disaster` in cm) bleiben als eigene Spalten unverändert erhalten, für alle Fälle, in denen der reale Meeresspiegel in cm benötigt wird.

Impact-Variablen (`Total Affected`, `Total Deaths`) werden nicht skaliert. Empirische Zählwerte in Personen sind direkt interpretierbar.

**Implementierung:**

In [42]:
msl_mean = sea_level_ts[SEA_VALUE].mean()
msl_std  = sea_level_ts[SEA_VALUE].std(ddof=0)

flood_linked["sea_level_at_start_z"]            = (flood_linked["sea_level_at_start"] - msl_mean) / msl_std
flood_linked["mean_sea_level_while_disaster_z"] = (flood_linked["mean_sea_level_while_disaster"] - msl_mean) / msl_std

print(f"Referenz-Meeresspiegel: μ = {msl_mean:.4f} cm, σ = {msl_std:.4f} cm")
print(flood_linked[["sea_level_at_start", "sea_level_at_start_z"]].describe().round(3))

Referenz-Meeresspiegel: μ = 5.9631 cm, σ = 3.2936 cm
       sea_level_at_start  sea_level_at_start_z
count             292.000               292.000
mean                6.471                 0.154
std                 3.565                 1.082
min                -0.417                -1.937
25%                 3.609                -0.715
50%                 6.326                 0.110
75%                 9.119                 0.958
max                14.521                 2.598


### Finaler Datensatz: `flood_linked`

In [43]:
print(f"flood_linked: {flood_linked.shape[0]} Ereignisse, {flood_linked.shape[1]} Spalten")
print(f"Fehlende sea_level_at_start: {flood_linked['sea_level_at_start'].isna().sum()}")
display(flood_linked.head(10))

flood_linked: 292 Ereignisse, 31 Spalten
Fehlende sea_level_at_start: 0


,DisNo.,ISO,Country,Region,Disaster Subtype,Origin,Latitude,Longitude,Total Affected,Total Deaths,is_outlier_Total Affected,is_outlier_Total Deaths,is_outlier,start_date,end_date,start_date_quality,end_date_quality,season,event_duration_days,has_coordinates,sea_level_at_start,sea_trend_at_start,sea_level_at_end,mean_sea_level_while_disaster,sea_level_lag_1d,sea_level_lag_2d,sea_level_lag_3d,sea_level_lag_4d,sea_level_lag_5d,sea_level_at_start_z,mean_sea_level_while_disaster_z
0,1999-0450-FRA,FRA,France,Europe,Riverine flood,Brief torrential rain,NaN,NaN,3005.0,36.0,False,False,False,1999-11-12,1999-11-15,complete,complete,autumn,4,False,1.313993,2.293326,1.229269,1.271478,1.342794,1.371823,1.401049,1.430440,1.459967,-1.411555,-1.424464
1,2000-0708-GRC,GRC,Greece,Europe,Flood (General),Extreme rain,NaN,NaN,600.0,NaN,False,False,False,2000-10-31,2000-10-31,complete,complete,autumn,1,False,2.305542,2.468741,2.305542,2.305542,2.235074,2.165319,2.096304,2.028052,1.960584,-1.110504,-1.110504
2,2023-0866-FRA,FRA,France,Europe,Flood (General),Heavy rains,NaN,NaN,6050.0,1.0,False,False,False,2023-12-30,2024-01-03,complete,complete,winter,5,False,14.521065,10.965430,14.527058,14.524420,14.518651,14.515857,14.512674,14.509099,14.505121,2.598336,2.599355
3,2022-0825-HRV,HRV,Croatia,Europe,Flood (General),NaN,NaN,NaN,NaN,NaN,False,False,False,2022-12-11,2022-12-11,complete,complete,winter,1,False,12.033220,10.400740,12.033220,12.033220,12.048471,12.062099,12.074087,12.084422,12.093092,1.842984,1.842984
4,2022-0825-BIH,BIH,Bosnia and Herzegovina,Europe,Flood (General),NaN,NaN,NaN,3000.0,1.0,False,False,False,2022-12-11,2022-12-22,complete,complete,winter,12,False,12.033220,10.400740,11.764241,11.912309,12.048471,12.062099,12.074087,12.084422,12.093092,1.842984,1.806273
5,2014-0164-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rains,NaN,NaN,1000000.0,25.0,True,False,True,2014-05-12,2014-05-22,complete,complete,spring,11,False,5.573401,6.427430,5.778097,5.671382,5.556266,5.539779,5.523946,5.508777,5.494277,-0.118327,-0.088578
6,2001-0642-GRC,GRC,Greece,Europe,Flash flood,Heavy rain,NaN,NaN,600.0,NaN,False,False,False,2001-11-29,2001-11-29,complete,complete,autumn,1,False,0.013958,2.680990,0.013958,0.013958,0.038181,0.063944,0.091210,0.119938,0.150084,-1.806268,-1.806268
7,2001-0802-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rain,NaN,NaN,9000.0,NaN,False,False,False,2001-06-20,2001-06-21,complete,complete,summer,2,False,1.327703,2.591551,1.331541,1.329622,1.324738,1.322653,1.321453,1.321145,1.321733,-1.407393,-1.406810
8,2019-0194-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rain,NaN,NaN,600.0,1.0,False,False,False,2019-05-12,2019-05-13,complete,complete,spring,2,False,6.782757,8.603725,6.793387,6.788072,6.772382,6.762283,6.752481,6.742994,6.733842,0.248854,0.250468
9,2002-0764-GRC,GRC,Greece,Europe,Riverine flood,Heavy rain,NaN,NaN,NaN,NaN,False,False,False,2002-12-06,2002-12-10,complete,complete,winter,5,False,5.174124,2.897837,5.076468,5.126674,5.195095,5.214692,5.232920,5.249783,5.265288,-0.239554,-0.253961
